In [ ]:
# !pip install langchain-community pypdf
# !pip install langchain-huggingface
# !pip install sentence-transformers
# !pip install -qU langchain-chroma

In [2]:
import re
import hashlib

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


In [ ]:

def deterministic_chunk_id(source, page, chunk_text):
    payload = f"{source}|{page}|{chunk_text.strip().lower()}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


In [ ]:
"""Carrega o PDF e enriquece os metadados de cada página."""

file_path = "/home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/data/pablo/sindilojas_2025_2026.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()
# docs = enrich_metadata(docs, file_path)

print(f"Páginas carregadas: {len(docs)}")
print("Exemplo de metadata:", docs[0].metadata)


Páginas carregadas: 38
Exemplo de metadata: {'producer': 'Microsoft® Word para Microsoft 365 - by Lacuna Software PKI SDK', 'creator': 'Microsoft® Word para Microsoft 365', 'creationdate': '2025-11-24T18:02:52-03:00', 'documentkey': '5ccd231f-5127-4696-84df-b576074c6eba', 'author': 'Vivianne Morais', 'moddate': '2025-11-24T21:54:55+00:00', 'source': '/home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/data/pablo/sindilojas_2025_2026.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1'}


In [8]:
"""Split the loaded documents into smaller chunks and print the total number of chunks created."""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=80, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

# all_splits = structural_then_char_split(docs, chunk_size=600, chunk_overlap=80)

# print(f"Total de chunks: {len(all_splits)}")
# print("Exemplo de metadata do chunk:", all_splits[0].metadata)

225


In [9]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# embeddings = HuggingFaceEmbeddings(model_name="embaas/sentence-transformers-multilingual-e5-base")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9181.44it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[0.004808688070625067, -0.040888428688049316, -0.023646898567676544, 0.03210205212235451, 0.021650847047567368, -0.058194007724523544, 0.01024177111685276, -0.0009654280147515237, -0.05202673748135567, -0.023083871230483055]


In [11]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [12]:
chunk_ids = [
    deterministic_chunk_id(
        chunk.metadata["source"],
        chunk.metadata["page"],
        chunk.page_content
    )
    for chunk in all_splits
]

ids = vector_store.add_documents(documents=all_splits, ids=chunk_ids)

In [13]:
results = vector_store.similarity_search_with_score(
    "Quais as garantias ao retornar de férias?"
)

doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.6680964827537537

page_content='do equivalente às incidências sobre férias integrais e proporcionais  sempre acrescidas do 
terço constitucional, décimo terceiro salário integral e proporcional. 
 
43 - GARANTIA DE EMPREGO - RETORNO DAS FÉRIAS -  O empregado que retornar de 
férias não poderá ser dispensado antes de 30 (trinta) dias, contados  a partir do primeiro dia 
de trabalho. 
 
Parágrafo 1º - Na hipótese do fracionamento das férias individuais e/ou féri as coletivas, a 
garantia de emprego será proporcional aos dias de férias gozados ou abonados. Não serão' metadata={'documentkey': '5ccd231f-5127-4696-84df-b576074c6eba', 'section': 'Parágrafo único', 'creator': 'Microsoft® Word para Microsoft 365', 'title': '', 'page_label': '21', 'start_index': 1463, 'page': 20, 'total_pages': 38, 'producer': 'Microsoft® Word para Microsoft 365 - by Lacuna Software PKI SDK', 'moddate': '2025-11-24T21:54:55+00:00', 'author': 'Vivianne Morais', 'clause': 'cláusula 41.', 'source': 'D:\\UC